 # **Import Dependencies and Page Setup**

 Modified to remove 'ee' and add 'geopy' for offline city geocoding.

In [7]:
%%writefile app.py


# Import Dependencies

import os
import pandas as pd
import numpy as np
import folium
from streamlit_folium import folium_static
import joblib
import streamlit as st
from geopy.geocoders import Nominatim



Overwriting app.py


 # **Page Configuration**

 Cleaned block. Google Earth Engine authentication and initialization removed.

In [8]:
%%writefile -a app.py


# Page configuration

st.set_page_config(page_title="Carbon Prediction Dashboard", layout="wide")



Appending to app.py


 # **Spatial Customization Parameters**

 Preserved for layout configuration.

In [9]:
%%writefile -a app.py


# Color palettes and graphics configuration
dicionario_cores = {
    1: "#32a65e", 3: "#1f8d49", 4: "#7dc975", 5: "#04381d", 6: "#026975",
    49: "#02d659", 10: "#ad975a", 11: "#519799", 12: "#d6bc74", 32: "#fc8114",
    29: "#ffaa5f", 50: "#ad5100", 13: "#d89f5c", 14: "#FFFFB2", 15: "#edde8e",
    18: "#E974ED", 19: "#C27BA0", 39: "#f5b3c8", 20: "#db7093", 40: "#c71585",
    62: "#ff69b4", 41: "#f54ca9", 36: "#d082de", 46: "#d68fe2", 47: "#9932cc",
    48: "#e6ccff", 9: "#7a5900", 21: "#ffefc3", 22: "#d4271e", 23: "#ffa07a",
    24: "#d4271e", 30: "#9c0027", 25: "#db4d4f", 26: "#0000FF", 33: "#2532e4",
    31: "#091077", 27: "#ffffff"
}

dicionario_classes = {
    1: "Floresta", 3: "Formação Florestal", 4: "Formação Savânica", 5: "Mangue",
    6: "Floresta Alagável (beta)", 49: "Restinga Arborizada", 10: "Formação Natural não Florestal",
    11: "Campo Alagado e Área Pantanosa", 12: "Formação Campestre", 32: "Apicum",
    29: "Afloramento Rochoso", 50: "Restinga Herbácea", 13: "Outras Formações não Florestais",
    14: "Agropecuária", 15: "Pastagem", 18: "Agricultura", 19: "Lavoura Temporária",
    39: "Soja", 20: "Cana", 40: "Arroz", 62: "Algodão (beta)", 41: "Outras Lavouras Temporárias",
    36: "Lavoura Perene", 46: "Café", 47: "Citrus", 48: "Outras Lavouras Perenes",
    9: "Silvicultura", 21: "Mosaico de Usos", 22: "Área não Vegetada", 23: "Praia, Duna e Areal",
    24: "Área Urbanizada", 30: "Mineração", 25: "Outras Áreas não Vegetadas", 26: "Corpo D'água",
    33: "Rio, Lago e Oceano", 31: "Aquicultura", 27: "Não observado"
}
## Pallete
paleta_nomes = {key:value for key, value in zip(dicionario_classes.values(), dicionario_cores.values())}
paleta_nomes

# Creating the color palette
paleta_cores = {
    0: "#ffffff",1: "#32a65e", 2: "#32a65e",3: "#1f8d49",
    4: "#7dc975",5: "#04381d", 6: "#026975",7: "#000000",
    8: "#000000",9: "#7a6c00", 10: "#ad975a",11: "#519799",
    12: "#d6bc74",13: "#d89f5c", 14: "#FFFFB2",15: "#edde8e",
    16: "#000000",17: "#000000", 18: "#f5b3c8",19: "#C27BA0",
    20: "#db7093",21: "#ffefc3", 22: "#db4d4f",23: "#ffa07a",
    24: "#d4271e",25: "#db4d4f", 26: "#0000FF",27: "#000000",
    28: "#000000",29: "#ffaa5f", 30: "#9c0027",31: "#091077",
    32: "#fc8114",33: "#2532e4", 34: "#93dfe6",35: "#9065d0",
    36: "#d082de",37: "#000000", 38: "#000000",39: "#f5b3c8",
    40: "#c71585",41: "#f54ca9", 42: "#cca0d4",43: "#dbd26b",
    44: "#807a40",45: "#e04cfa", 46: "#d68fe2",47: "#9932cc",
    48: "#e6ccff",49: "#02d659", 50: "#ad5100",51: "#000000",
    52: "#000000",53: "#000000", 54: "#000000",55: "#000000",
    56: "#000000",57: "#CC66FF", 58: "#FF6666",59: "#006400",
    60: "#8d9e8b",61: "#f5d5d5", 62: "#ff69b4"
}
palette_list = list(paleta_cores.values())
len(palette_list)
vis_gpp = {
    'min': 0,
    'max': 3000, # Valor comum de gC/m2/ano em pastagens produtivas
    'palette': ['#f7fcb9', '#addd8e', '#31a354'] # Do amarelo claro ao verde escuro
}


Appending to app.py


 # **Load Data & ML Model (Cached)**

In [10]:
%%writefile -a app.py


# Load Data and ML model

@st.cache_data
def load_csv():
    csv_path = "/content/drive/MyDrive/Colab Notebooks/files/output/final/data_integration_all_areas_final.csv"
    return pd.read_csv(csv_path)

@st.cache_resource
def load_ml_model():
    model_path = '/content/drive/MyDrive/Colab Notebooks/files/extra/carbon_tree_model.pkl'
    try:
        return joblib.load(model_path)
    except Exception as e:
        st.sidebar.error(f"Model Load Error: {e}")
        return None

all_farms_df = load_csv()
ml_model = load_ml_model()



Appending to app.py


 # **Dashboard Interface & Spatial Processing (Approximation Mode)**

 Replaced Earth Engine imagery processing with live city geocoding and calculated area circles.

In [11]:
%%writefile -a app.py


# Sidebar / user interface

st.sidebar.header("📍 Location Filters")

list_state = sorted(list(all_farms_df['state_name'].unique()))
state_name = st.sidebar.selectbox('State:', list_state)

list_cities = sorted(list(all_farms_df[all_farms_df['state_name'] == state_name]['city_name'].unique()))
city_name = st.sidebar.selectbox('City:', list_cities)

list_farms = sorted(list(all_farms_df[all_farms_df['city_name'] == city_name]['farm'].unique()))

# Adding index=None and a placeholder prevents the first farm from being automatically loaded.
farm_id = st.sidebar.selectbox('Farm (ID):', list_farms, index=None, placeholder="Select a farm...")


# Metrics and AI inference

if farm_id is None:
    
    # Landing page
    
    st.title("🌱 Carbon Prediction Dashboard")
    st.markdown("---")
    st.markdown("### 📊 Welcome to the MRV Carbon Monitoring System")
    st.markdown("This platform integrates agricultural land data with Machine Learning to predict and analyze carbon balances (emissions and sequestration) across Brazilian farms, supporting Measurement, Reporting, and Verification (MRV) processes.")
    
    st.markdown("#### 🔍 How it works")
    st.markdown("1. **Location Filters:** Use the sidebar on the left to navigate through States, Cities, and specific Farm IDs.")
    st.markdown("2. **Carbon Metrics:** Once a farm is selected, the system compares the historical reality (tCO2/ha) against the Artificial Intelligence prediction.")
    st.markdown("3. **Spatial Context:** A dynamic map generates a proportional boundary based on the farm's area size to provide spatial awareness.")
    
    st.markdown("#### 🎯 Project Objective")
    st.markdown("This research aims to evaluate Machine Learning models linked to carbon mitigation in the agricultural sector. It provides a visual and interactive interface to validate predictive models and strengthen robust MRV protocols.")
    st.markdown("---")

else:
    
    # Screen of the specific farm
    
    data = all_farms_df[all_farms_df['farm'] == farm_id].iloc[0]
    
    st.title(f"🌱 Predicted Carbon Dashboard - Farm {farm_id}")
    st.markdown("---")
    
    ai_prediction_str = "Unavailable"
    if ml_model is not None:
        try:
            colunas_treino = [
                'area_size', 
                'biome_cod', 
                'state_cod', 
                'climate_cod', 
                'city_cod', 
                'year'
            ]
            features = data[colunas_treino].values.reshape(1, -1)
            prediction_value = ml_model.predict(features)[0]
            ai_prediction_str = f"{prediction_value:.4f}"
        except Exception as ml_err:
            ai_prediction_str = "Inference Error"
            st.error(f"Erro técnico do ML: {ml_err}")

    # Vertical display of metrics
    st.metric("Area (ha)", f"{data['area_size']:.2f}")
    st.metric("Biome", data['biome_name'])
    st.metric("tCO2/ha (Historical Reality)", f"{data['balance_CO2_ha']:.4f}")
    st.metric("tCO2/ha (Predicted AI)", ai_prediction_str)
    st.metric("Total CO2", f"{data['balance_CO2_area']:.2f}")
    
    st.markdown("---")
    
    # local geoprocessing and folium map
    with st.spinner('Calculating approximate farm location'):
        try:
            geolocator = Nominatim(user_agent="carbon_dashboard_ufjf")
            location_query = f"{city_name}, {state_name}, Brazil"
            location = geolocator.geocode(location_query, timeout=10)
            
            if location:
                center_lat = location.latitude
                center_lon = location.longitude
            else:
                center_lat = -14.2350
                center_lon = -51.9253

            M = folium.Map(location=[center_lat, center_lon], zoom_start=11, control_scale=True)

            area_ha = float(data['area_size'])
            radius_meters = np.sqrt((area_ha * 10000) / np.pi)

            folium.Circle(
                location=[center_lat, center_lon],
                radius=radius_meters,
                color="#32a65e",
                fill=True,
                fill_color="#7dc975",
                fill_opacity=0.5,
                popup=f"Farm {farm_id} - Estimated limits based on {area_ha:.2f} hectares."
            ).add_to(M)

            folium.Marker(
                location=[center_lat, center_lon],
                popup=f"Reference City Center: {city_name}",
                icon=folium.Icon(color="green", icon="info-sign")
            ).add_to(M)

            folium_static(M, width=700, height=500)

            st.info(f"ℹ️ Map Display: Showing a calculated proportional boundary circle of {radius_meters:.1f} meters around the center of {city_name} to represent the farm size.")

        except Exception as e:
            st.error(f"Error rendering the approximation map: {e}")

Appending to app.py


 # **Cloud Deployment Execution**

In [ ]:
# Install local dependencies adding 'geopy'
!pip install -q streamlit==1.29.0 folium streamlit-folium geopy

# Install web tunnel tool
!npm install -g localtunnel

# Fetch the Password/IP for authentication
import urllib
print("Key to access the website", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

# Run the dashboard
!streamlit run app.py --server.enableCORS false --server.enableXsrfProtection false & lt --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
changed 22 packages in 3s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋Key to access the website 35.252.254.42
your url is: https://lazy-guests-reply.loca.lt



  You can now view your Streamlit app in your browser.

  Network URL: http://172.28.0.12:8501
  External URL: http://35.252.254.42:8501

/content/app.py:206: DeprecationWarning: 
folium_static is deprecated and will be removed in a future release, or
simply replaced with with st_folium which always passes
returned_objects=[] to the component.
Please try using st_folium instead, and
post an issue at https://github.com/randyzwitch/streamlit-folium/issues
if you experience issues with st_folium.

  folium_static(M, width=700, height=500)


In [ ]:
#!rm -f app.py            # Deleta o arquivo de código velho
#!pkill -f streamlit      # Mata qualquer servidor do Streamlit que tenha ficado rodando "por baixo dos panos"